In [2]:
import sys
sys.path.append('../') # Adding anigen to the path.

import trimesh
import numpy as np
from pathlib import Path

from animgen.core.models.model import BaseModelClass

In [3]:
SEGMENTED_PATHS = {
    "snake_complete": Path("../generated_data/models/paint_mesh_Sea_Snake.glb"),
    "snake_complete_untextured": Path("../generated_data/models/img_mesh_Sea_Snake.glb"),
}

In [18]:
snake_complete = BaseModelClass(SEGMENTED_PATHS["snake_complete"]).mesh
snake_complete_untextured = BaseModelClass(SEGMENTED_PATHS["snake_complete_untextured"]).mesh
snake_complete.show()

Rendering Multiviews...: 100%|██████████| 20/20 [00:06<00:00,  3.29it/s]


In [5]:
import numpy as np

mesh = snake_complete.copy()  # Create a copy of the mesh to avoid modifying the original

edges = mesh.edges_sorted

unique_edges, counts = np.unique(
    edges,
    axis=0,
    return_counts=True,
)

boundary_edges = unique_edges[counts == 1]
nonmanifold_edges = unique_edges[counts > 2]

print("Vertices:", len(mesh.vertices))
print("Faces:", len(mesh.faces))
print("Watertight:", mesh.is_watertight)
print("Winding consistent:", mesh.is_winding_consistent)
print("Euler number:", mesh.euler_number)
print("Body count:", mesh.body_count)

print("Boundary edges:", len(boundary_edges))
print("Non-manifold edges:", len(nonmanifold_edges))

Vertices: 6993
Faces: 11126
Watertight: False
Winding consistent: True
Euler number: 54
Body count: 63
Boundary edges: 2752
Non-manifold edges: 0


In [6]:
mesh = snake_complete.copy()

print("Before")
print("Vertices:", len(mesh.vertices))
print("Watertight:", mesh.is_watertight)

# Weld coincident vertices
mesh.merge_vertices()

# Remove bad/redundant faces
mesh.update_faces(mesh.unique_faces())
mesh.update_faces(mesh.nondegenerate_faces())

# Clean vertices left behind
mesh.remove_unreferenced_vertices()

# Repair orientation
trimesh.repair.fix_winding(mesh)
trimesh.repair.fix_normals(mesh)

print("\nAfter")
print("Vertices:", len(mesh.vertices))
print("Faces:", len(mesh.faces))
print("Watertight:", mesh.is_watertight)
print("Winding:", mesh.is_winding_consistent)
print("Volume:", mesh.is_volume)

Before
Vertices: 6993
Watertight: False

After
Vertices: 6993
Faces: 11126
Watertight: False
Winding: True
Volume: False


In [7]:
edges = mesh.edges_sorted

_, counts = np.unique(
    edges,
    axis=0,
    return_counts=True
)

print("Boundary edges:", np.sum(counts == 1))
print("Normal edges:", np.sum(counts == 2))
print("Non-manifold edges:", np.sum(counts > 2))

Boundary edges: 2752
Normal edges: 15313
Non-manifold edges: 0


In [8]:
components = mesh.split(only_watertight=False)

print("Components:", len(components))

components = sorted(
    components,
    key=lambda m: m.area,
    reverse=True
)

for i, component in enumerate(components[:20]):
    print(
        i,
        "vertices =", len(component.vertices),
        "faces =", len(component.faces),
        "area =", component.area,
        "watertight =", component.is_watertight,
    )

Components: 63
0 vertices = 837 faces = 1483 area = 0.09578436101945145 watertight = False
1 vertices = 632 faces = 1059 area = 0.06629537135660715 watertight = False
2 vertices = 531 faces = 898 area = 0.05847168193773997 watertight = False
3 vertices = 267 faces = 425 area = 0.03330153499943049 watertight = False
4 vertices = 253 faces = 414 area = 0.03255872534120709 watertight = False
5 vertices = 277 faces = 437 area = 0.030062129204897384 watertight = False
6 vertices = 352 faces = 585 area = 0.027164571636221393 watertight = False
7 vertices = 206 faces = 331 area = 0.027138615674654344 watertight = False
8 vertices = 317 faces = 528 area = 0.026437226926011376 watertight = False
9 vertices = 241 faces = 379 area = 0.024481168267926644 watertight = False
10 vertices = 209 faces = 341 area = 0.022005209078865728 watertight = False
11 vertices = 158 faces = 252 area = 0.0207118434601968 watertight = False
12 vertices = 227 faces = 387 area = 0.01878189342474608 watertight = False


In [9]:
for i, component in enumerate(components):
    extent = component.extents

    print(
        f"{i:2d}",
        f"faces={len(component.faces):5d}",
        f"area={component.area:.6f}",
        f"extent={extent}",
    )

 0 faces= 1483 area=0.095784 extent=[1.19107182 0.05675582 0.34675057]
 1 faces= 1059 area=0.066295 extent=[0.35532807 0.04034145 1.07983684]
 2 faces=  898 area=0.058472 extent=[0.36104674 0.04670106 0.94121002]
 3 faces=  425 area=0.033302 extent=[0.80828511 0.06724669 0.15641053]
 4 faces=  414 area=0.032559 extent=[0.60830426 0.06062729 0.16463281]
 5 faces=  437 area=0.030062 extent=[0.8260017  0.04526301 0.1836676 ]
 6 faces=  585 area=0.027165 extent=[0.38174046 0.0413119  0.33431184]
 7 faces=  331 area=0.027139 extent=[0.30086242 0.0712877  0.52102924]
 8 faces=  528 area=0.026437 extent=[0.39646161 0.07595811 0.34076503]
 9 faces=  379 area=0.024481 extent=[0.34138824 0.03770889 0.77472958]
10 faces=  341 area=0.022005 extent=[0.1401601  0.06389563 0.55535655]
11 faces=  252 area=0.020712 extent=[0.17398648 0.04889136 0.5975072 ]
12 faces=  387 area=0.018782 extent=[0.25456591 0.07570895 0.22399806]
13 faces=  371 area=0.017927 extent=[0.58482165 0.02799371 0.20332214]
14 fac

In [10]:
def global_PCA(mesh: trimesh.Trimesh) -> tuple[np.ndarray, np.ndarray]:
    """
    Computes the global PCA of a mesh and returns the principal components.
    
    Parameters:
    mesh (trimesh.Trimesh): The input mesh for which to compute the PCA.
    
    Returns:
    np.ndarray: The principal components of the mesh.
    """
    vertices = mesh.vertices
    
    centered_vertices = vertices - np.mean(vertices, axis=0)
    
    covariance_matrix = np.cov(centered_vertices, rowvar=False)
    
    eigenvalues, eigenvectors = np.linalg.eig(covariance_matrix)
    
    sorted_indices = np.argsort(eigenvalues)[::-1]
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices]
    
    return sorted_eigenvectors, sorted_eigenvalues

In [ ]:
snake_complete.is_watertight, snake_complete.is_winding_consistent, snake_complete.is_volume

(False, True, False)

In [12]:
import numpy as np

edges = snake_complete.edges_sorted

unique_edges, counts = np.unique(
    edges,
    axis=0,
    return_counts=True
)

boundary_edges = unique_edges[counts == 1]
nonmanifold_edges = unique_edges[counts > 2]

print("Vertices:", len(snake_complete.vertices))
print("Faces:", len(snake_complete.faces))
print("Boundary edges:", len(boundary_edges))
print("Non-manifold edges:", len(nonmanifold_edges))

Vertices: 6993
Faces: 11126
Boundary edges: 2752
Non-manifold edges: 0


In [13]:
import networkx as nx

G = nx.Graph()
G.add_edges_from(boundary_edges)

components = list(nx.connected_components(G))

sizes = sorted(
    [len(c) for c in components],
    reverse=True
)

print("Number of boundary components:", len(components))
print("Boundary component sizes:", sizes)

Number of boundary components: 68
Boundary component sizes: [203, 189, 162, 134, 117, 115, 107, 101, 101, 90, 89, 83, 83, 79, 75, 67, 65, 62, 62, 61, 61, 50, 49, 49, 46, 44, 43, 38, 34, 33, 29, 20, 17, 16, 14, 12, 11, 9, 9, 8, 8, 6, 6, 6, 5, 5, 5, 5, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]


In [14]:
import numpy as np
import networkx as nx

V = snake_complete.vertices

G = nx.Graph()
G.add_edges_from(boundary_edges)

components = list(nx.connected_components(G))

mesh_diag = np.linalg.norm(
    snake_complete.bounds[1] - snake_complete.bounds[0]
)

info = []

for component in components:
    ids = np.asarray(list(component))
    points = V[ids]

    extent = np.linalg.norm(
        points.max(axis=0) - points.min(axis=0)
    )

    info.append({
        "vertices": len(ids),
        "extent": extent,
        "relative_extent": extent / mesh_diag
    })

info = sorted(
    info,
    key=lambda x: x["relative_extent"],
    reverse=True
)

for x in info:
    print(x)

{'vertices': 189, 'extent': np.float64(1.241719929970822), 'relative_extent': np.float64(0.6031142047400069)}
{'vertices': 203, 'extent': np.float64(1.1374999077934738), 'relative_extent': np.float64(0.5524936305860959)}
{'vertices': 162, 'extent': np.float64(1.0091412269685447), 'relative_extent': np.float64(0.4901486993027407)}
{'vertices': 101, 'extent': np.float64(0.8474513655134842), 'relative_extent': np.float64(0.4116145227527338)}
{'vertices': 115, 'extent': np.float64(0.8473543002489584), 'relative_extent': np.float64(0.4115673773068008)}
{'vertices': 107, 'extent': np.float64(0.8260212981986641), 'relative_extent': np.float64(0.4012057520677)}
{'vertices': 90, 'extent': np.float64(0.6330984990369967), 'relative_extent': np.float64(0.30750146514742865)}
{'vertices': 62, 'extent': np.float64(0.624240752177226), 'relative_extent': np.float64(0.3031991802084695)}
{'vertices': 61, 'extent': np.float64(0.6213879460850572), 'relative_extent': np.float64(0.30181354739705424)}
{'verti

In [15]:
"""
NOTE: 
Decimation Experiments show o3d based decimation is not really good for our use case, 
as it does not preserve the mesh topology, thus tough to do mesh contraction.
"""

'\nNOTE: \nDecimation Experiments show o3d based decimation is not really good for our use case, \nas it does not preserve the mesh topology, thus tough to do mesh contraction.\n'

In [16]:
import numpy as np
import pymeshlab
import trimesh


def preprocess_mesh(mesh, decimation_percent=90):
    ms = pymeshlab.MeshSet()

    ms.add_mesh(
        pymeshlab.Mesh(
            vertex_matrix=np.asarray(mesh.vertices),
            face_matrix=np.asarray(mesh.faces),
        )
    )

    target_faces = int(len(mesh.faces) * (1-decimation_percent/100))

    ms.meshing_decimation_quadric_edge_collapse(
        targetfacenum=target_faces,
        preservetopology=True,
        preserveboundary=True,
    ) # Much better decimation as compared to other o3d decimation pipeline

    simplified = ms.current_mesh()

    result = trimesh.Trimesh(
        vertices=simplified.vertex_matrix(),
        faces=simplified.face_matrix(),
        process=False,
    )
    
    return result

In [20]:
snake_complete_untextured.is_watertight, snake_complete_untextured.is_winding_consistent, snake_complete_untextured.is_volume

(True, True, True)

In [21]:
preprocessed_mesh = preprocess_mesh(snake_complete_untextured, decimation_percent=90)
preprocessed_mesh.is_watertight, preprocessed_mesh.is_winding_consistent, preprocessed_mesh.is_volume

(True, True, True)

In [23]:
# Vertices and Faces

untextured_vertices = len(snake_complete_untextured.vertices)
untextured_faces = len(snake_complete_untextured.faces)

preprocessed_vertices = len(preprocessed_mesh.vertices)
preprocessed_faces = len(preprocessed_mesh.faces)

print(f"Untextured Mesh: Vertices = {untextured_vertices}, Faces = {untextured_faces}")
print(f"Preprocessed Mesh: Vertices = {preprocessed_vertices}, Faces = {preprocessed_faces}")

Untextured Mesh: Vertices = 55632, Faces = 111264
Preprocessed Mesh: Vertices = 5563, Faces = 11126


In [22]:
preprocessed_mesh.show()

In [ ]:
"""
NOTE:

o3d decimation does not preserve the mesh topology, which makes it difficult to perform mesh contraction.
But as seen above pymeshlab decimation preserves the mesh topology and thus default decimation is being shifted to that.
"""